# COMP9312 26T3 - Q2: connected k-core index

Implement `KCoreIndex.build()`, `query(k, v)` and `max_k(u, v)`. The constructor receives `n` and a simple undirected edge list; you may change internal representations while preserving signatures.

- `build()` preprocesses once and returns `self`.
- `query(k, v)` returns an ascending list of distinct vertex IDs; `[]` if absent.
- `k` is a non-negative integer. At `k == 0`, return the ordinary connected component.
- `max_k(u, v)` returns the largest shared threshold; `-1` if disconnected; the core number if `u == v` (0 for an isolated vertex).
- Inputs use IDs `0..n-1`. If `n == 0`, edges are empty and no vertex queries occur. Calls follow `build()`.

Use only the Python Standard Library. Put helpers/imports before the local-test cell. Read the accompanying Q2 specification and Project Summary.


In [1]:
class KCoreIndex:
    def __init__(self, n, edges):
        self.n = n
        self.adj = [[] for _ in range(n)]
        for u, v in edges:
            self.adj[u].append(v)
            self.adj[v].append(u)

    def _core_decomposition(self):
        """Return core numbers and a nondecreasing core order in O(n + m)."""
        n = self.n
        core = [len(row) for row in self.adj]
        max_degree = max(core, default=0)
        bins = [0] * (max_degree + 1)
        for degree in core:
            bins[degree] += 1

        # Counting sort by degree, with positions supporting O(1) moves
        # between adjacent buckets as neighbours are peeled.
        start = 0
        for degree in range(max_degree + 1):
            count = bins[degree]
            bins[degree] = start
            start += count
        order = [0] * n
        position = [0] * n
        for v, degree in enumerate(core):
            position[v] = bins[degree]
            order[position[v]] = v
            bins[degree] += 1
        for degree in range(max_degree, 0, -1):
            bins[degree] = bins[degree - 1]
        bins[0] = 0

        for v in order:
            for u in self.adj[v]:
                # Do not lower a neighbour below the current peeling level.
                if core[u] > core[v]:
                    degree = core[u]
                    old = position[u]
                    first = bins[degree]
                    other = order[first]
                    if u != other:
                        order[old], order[first] = other, u
                        position[other], position[u] = old, first
                    bins[degree] += 1
                    core[u] -= 1
        return core, order

    def build(self):
        core, order = self._core_decomposition()
        n = self.n
        dsu = list(range(n))
        size = [1] * n
        top = list(range(n))
        active = bytearray(n)

        def find(v):
            while dsu[v] != v:
                dsu[v] = dsu[dsu[v]]
                v = dsu[v]
            return v

        # Leaves are vertices, labelled with their core numbers. Each union
        # adds an internal node labelled with the current core threshold.
        parent = list(range(n))
        left = [-1] * n
        right = [-1] * n
        level = core.copy()
        leaf_count = [1] * n
        for v in reversed(order):
            active[v] = 1
            a = v  # Newly active v starts as a singleton DSU component.
            for u in self.adj[v]:
                if not active[u]:
                    continue
                b = find(u)
                if a == b:
                    continue
                node = len(parent)
                x, y = top[a], top[b]
                parent.append(node)
                parent[x] = parent[y] = node
                left.append(x)
                right.append(y)
                level.append(core[v])
                leaf_count.append(size[a] + size[b])
                if size[a] < size[b]:
                    a, b = b, a
                dsu[b] = a
                size[a] += size[b]
                top[a] = node

        # Heavy-first DFS gives contiguous heavy chains in preorder and a
        # contiguous leaf interval for every component. No recursion is used.
        count = len(parent)
        depth = [0] * count
        head = [0] * count
        position = [0] * count
        first = [0] * count
        preorder = []
        vertices = []
        component = [0] * n
        for root, p in enumerate(parent):
            if p != root:
                continue
            stack = [(root, root)]
            while stack:
                node, chain = stack.pop()
                head[node] = chain
                position[node] = len(preorder)
                preorder.append(node)
                first[node] = len(vertices)
                if node < n:
                    component[node] = root
                    vertices.append(node)
                    continue
                heavy, light = left[node], right[node]
                if leaf_count[heavy] < leaf_count[light]:
                    heavy, light = light, heavy
                depth[heavy] = depth[light] = depth[node] + 1
                stack.append((light, light))
                stack.append((heavy, chain))

        # Retain only O(n) index data. DSU, buckets and child lists are
        # construction workspace; the original adjacency remains unchanged.
        self.core = core
        self._parent = parent
        self._level = level
        self._leaf_count = leaf_count
        self._depth = depth
        self._head = head
        self._position = position
        self._first = first
        self._preorder = preorder
        self._vertices = vertices
        self._component = component
        return self

    def query(self, k, v):
        if self.core[v] < k:
            return []

        # Find the highest ancestor whose merge threshold is at least k.
        # Each chain jump crosses a light edge, so there are O(log n) jumps.
        node = v
        head, parent, level = self._head, self._parent, self._level
        while True:
            chain = head[node]
            if level[chain] < k:
                # The answer lies inside this chain. Thresholds are
                # nondecreasing from its head towards its tail; binary
                # search once for the first ancestor with threshold >= k.
                lo, hi = self._position[chain], self._position[node]
                preorder = self._preorder
                while lo < hi:
                    mid = (lo + hi) // 2
                    if level[preorder[mid]] >= k:
                        hi = mid
                    else:
                        lo = mid + 1
                node = preorder[lo]
                break
            node = chain
            p = parent[node]
            if p == node or level[p] < k:
                break
            node = p

        start = self._first[node]
        answer = self._vertices[start:start + self._leaf_count[node]]
        answer.sort()  # Sort a copy; callers cannot mutate the index.
        return answer

    def max_k(self, u, v):
        if u == v:
            return self.core[u]
        if self._component[u] != self._component[v]:
            return -1

        # Their LCA is the highest threshold at which the vertices meet.
        head, depth, parent = self._head, self._depth, self._parent
        while head[u] != head[v]:
            if depth[head[u]] > depth[head[v]]:
                u = parent[head[u]]
            else:
                v = parent[head[v]]
        ancestor = u if depth[u] < depth[v] else v
        return self._level[ancestor]


## Index design and correctness

1. **Core decomposition.** Peel vertices in nondecreasing degree order using
   degree buckets and constant-time swaps. When processing `v`, decrease a
   neighbour's stored degree only if it is larger than `core[v]`. These values
   represent remaining degrees capped below by the current peeling level, which
   never decreases. Immediately before a level `c` starts, the remaining graph
   has minimum degree at least `c`, so every vertex subsequently removed at that
   level belongs to a `c`-core. Such a vertex cannot belong to a `(c + 1)`-core:
   by induction, previously removed vertices cannot belong to that core, while
   at its removal it has at most `c` remaining neighbours. The final stored
   degrees are therefore the core numbers, and the returned order is
   nondecreasing in core number.
2. **Component merge forest.** Activate vertices in reverse core order. Join an
   active vertex to every already active neighbour using DSU. At each successful
   union, add a parent of the two component roots, labelled with the current
   vertex's core number. Original vertices are leaves labelled with their own
   core numbers. After a whole level `k` is processed, the active graph is exactly
   the graph induced by vertices with core number at least `k`, which is the
   `k`-core. DSU thus represents its connected components. Equal-level unions
   are all retained; a query climbs through all of them, so tie order cannot
   change a result. There are at most `n - 1` successful unions.
3. **Linear-space navigation and enumeration.** Store a heavy-light decomposition
   of this binary forest. The child with more leaves is heavy; taking a light
   edge at least halves the number of leaves. Heavy-first iterative DFS stores
   each heavy chain consecutively in preorder and each subtree's leaves in one
   contiguous interval. No component membership lists are duplicated across
   thresholds, and no recursion or binary-lifting table is needed.

For `query(k, v)`, reject `v` if its core number is below `k`. Otherwise its
connected `k`-core is the highest ancestor of its leaf with label at least `k`:
every union up to that node is present at threshold `k`, and the next union, if
any, is below the threshold. Jump entire heavy chains while their labels allow
it. Labels are nondecreasing from the head to the tail of a chain, so at most one
binary search finds the final ancestor inside a chain. Copy its leaf interval
and sort the copy to obtain distinct vertex IDs in ascending order. At `k = 0`,
all vertices are eligible, including isolated vertices, and this selects the
ordinary connected component.

For distinct `u` and `v` in the same tree, their LCA is the first common component
as the activation threshold decreases. Its label is the largest threshold at
which they are connected, so it answers `max_k(u, v)`. Different tree roots give
`-1`, and identical endpoints return their own core number.

## Complexity

Let `n` and `m` be vertex and edge counts, `d(v)` the original degree,
`k_max` the largest core number, `a = |A|` the answer size, and `alpha(n)` the
inverse Ackermann function (at least 1 for small `n`).

- **Constructor:** `O(n + m)` time and adjacency space.
- **Core decomposition:** `O(n + m)` time, since bucket setup costs
  `O(n + max_v d(v)) = O(n)` for a simple graph, and neighbour scans total
  `sum_v d(v) = 2m`. Each bucket update is constant time.
- **Remaining index construction:** `O(n + m alpha(n))` time. Reverse core
  order is already available, so no vertex or edge sorting is required. The
  activation pass examines `2m` adjacency entries, processes each edge once
  when both endpoints are active, and creates only `O(n)` merge nodes.
  Heavy-light decomposition and leaf intervals require `O(n)` further work.
  **Total construction, including decomposition, is `O(n + m alpha(n))`.**
  In particular it does not repeat a graph scan for each of the `k_max` levels.
- **`query(k, v)`:** `O(1)` for an absent vertex; otherwise `O(log(n + 1))`
  index access (light-edge jumps plus at most one chain binary search),
  `O(a)` copying and `O(a log(a + 1))` sorting. Total:
  `O(log(n + 1) + a log(a + 1))`. No original-graph edges are examined by a query.
- **`max_k(u, v)`:** `O(log(n + 1))` worst case for heavy-light LCA;
  `O(1)` for identical or disconnected endpoints.
- **Space:** the original adjacency uses `O(n + m)`. Additional retained
  index space is `O(n)`, separate from adjacency and any caller-owned input
  edge list. Peak construction workspace beyond the graph is also `O(n)`:
  degree buckets, DSU, binary children, traversal stacks and index arrays.
  The returned membership list uses `O(a)` space, and sorting may need
  `O(a)` temporary workspace. Index access itself and `max_k` use `O(1)`
  additional workspace. Empty graphs are supported, and `build()` returns `self`.


## Local tests
Run after implementing. Do not change this cell. Timings are informational, not fixed grading limits. The test generator and full-output digests are deterministic.

In [2]:
# LOCAL TESTS - keep this marker and the tests unchanged.
"""Deterministic fixtures and independent oracles; Standard Library only."""
import hashlib
import heapq
from collections import deque


class FixedRandom:
    """Explicit xorshift32 test generator, independent of random module versions."""
    def __init__(self, seed):
        self.state = seed & 0xffffffff or 1

    def below(self, limit):
        x = self.state
        x ^= (x << 13) & 0xffffffff
        x ^= x >> 17
        x ^= (x << 5) & 0xffffffff
        self.state = x & 0xffffffff
        return self.state % limit


def graph(n, m, seed=1):
    assert 0 <= m <= n * (n - 1) // 2
    rng, seen, edges = FixedRandom(seed), set(), []
    while len(edges) < m:
        u, v = rng.below(n), rng.below(n)
        if u == v:
            continue
        edge = (min(u, v), max(u, v))
        if edge not in seen:
            seen.add(edge)
            edges.append(edge)  # Never enumerate the set.
    return edges


def pairs(n, count, seed=2):
    rng = FixedRandom(seed)
    return [(rng.below(n), rng.below(n)) for _ in range(count)]


def query_pairs(n, count, seed=3):
    rng = FixedRandom(seed)
    return [(rng.below(13), rng.below(n)) for _ in range(count)]


def add_digest(digest, values):
    digest.update((",".join(map(str, values)) + "\n").encode("ascii"))


def digest_rows(rows):
    digest = hashlib.sha256()
    for row in rows:
        add_digest(digest, row)
    return digest.hexdigest()


def brute_q1(n, edges, queries):
    answers = []
    for a, b in queries:
        if a == b:
            answers.append(0)
            continue
        adj, answer = [set() for _ in range(n)], -1
        for t, (u, v) in enumerate(edges, 1):
            adj[u].add(v)
            adj[v].add(u)
            seen, pending = {a}, [a]
            while pending:
                u = pending.pop()
                for v in adj[u] - seen:
                    seen.add(v)
                    pending.append(v)
            if b in seen:
                answer = t
                break
        answers.append(answer)
    return answers


def brute_q2(n, edges, k, v):
    adj = [set() for _ in range(n)]
    for a, b in edges:
        adj[a].add(b)
        adj[b].add(a)
    alive, degree = [True] * n, [len(row) for row in adj]
    pending = deque(x for x in range(n) if degree[x] < k)
    while pending:
        x = pending.popleft()
        if not alive[x]:
            continue
        alive[x] = False
        for y in adj[x]:
            if alive[y]:
                degree[y] -= 1
                if degree[y] < k:
                    pending.append(y)
    if not alive[v]:
        return []
    seen, pending = {v}, [v]
    while pending:
        x = pending.pop()
        for y in adj[x]:
            if alive[y] and y not in seen:
                seen.add(y)
                pending.append(y)
    return sorted(seen)


def brute_max(n, edges, u, v):
    answer = -1
    for k in range(n + 1):
        if v in brute_q2(n, edges, k, u):
            answer = k
        else:
            break
    return answer



import time
edges = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3),(3,4),(4,5),(3,5),(5,6),(6,7),(6,8),(6,9),(7,8),(7,9),(8,9),(9,10)]
idx = KCoreIndex(11,edges).build()
assert idx.query(1,0)==list(range(11)) and idx.query(2,4)==list(range(10))
assert idx.query(3,3)==[0,1,2,3] and idx.query(3,7)==[6,7,8,9]
assert idx.query(4,0)==[] and idx.query(2,10)==[]
assert [idx.max_k(u,v) for u,v in [(0,7),(0,4),(4,10),(1,2)]]==[2,2,1,3]
idx = KCoreIndex(5,[(0,1),(1,2),(0,2)]).build()
assert idx.query(0,4)==[4] and idx.query(1,4)==[]
assert idx.query(0,0)==[0,1,2] and idx.query(99,0)==[]
assert idx.max_k(4,4)==0 and idx.max_k(3,4)==-1 and idx.max_k(0,0)==2
KCoreIndex(0,[]).build()
rng = FixedRandom(9312)
for trial in range(150):
    n = 2 + rng.below(11)
    E = graph(n,rng.below(min(24,n*(n-1)//2)+1),trial+100)
    idx = KCoreIndex(n,E).build()
    for _ in range(6):
        k,v = rng.below(6),rng.below(n)
        assert idx.query(k,v)==brute_q2(n,E,k,v), (n,E,k,v)
    for u,v in pairs(n,6,trial+1000)+[(0,0)]:
        assert idx.max_k(u,v)==brute_max(n,E,u,v), (n,E,u,v)
print("Q2 figure, boundary cases and 150 oracle trials passed")
n = 10000
E = graph(n,80000,1)
start = time.perf_counter()
idx = KCoreIndex(n,E).build()
build_time = time.perf_counter()-start
digest = hashlib.sha256()
start = time.perf_counter()
for k,v in query_pairs(n,2000,3):
    answer = idx.query(k,v)
    assert type(answer) is list
    assert all(type(value) is int for value in answer)
    add_digest(digest,answer)
query_time = time.perf_counter()-start
assert digest.hexdigest()=='f303322c378ce85a9afee5ad04ba6735c36dddb7ff95a360ebfea2cd65b35b88', "Q2 full-membership digest mismatch"
start = time.perf_counter()
answers = [idx.max_k(u,v) for u,v in pairs(n,20000,4)]
max_time = time.perf_counter()-start
assert all(type(value) is int for value in answers)
assert digest_rows([answers])=='39ce2689f77559bdbf3079ee376211c4dfc95a3e5c52b00f7fb22fcb6c88f549', "Q2 max_k digest mismatch"
print("Q2 scale passed; build %.3f s, queries+digest %.3f s, max_k %.3f s" % (build_time,query_time,max_time))


Q2 figure, boundary cases and 150 oracle trials passed


Q2 scale passed; build 0.075 s, queries+digest 4.311 s, max_k 0.025 s
